# CineMatch-AI — Neural modeling

Rating prediction (RMSE / MAE on 0.5–5.0 stars). Each section below explains one model and shows **full-pipeline metrics** from a completed run on all 22.4M training rows (~9 hours on CPU).

| Model | Signal | Idea |
|-------|--------|------|
| **Baseline** | — | Global mean |
| **GMF** | Collaborative | Dot product of user & movie embeddings (linear CF) |
| **NeuMF** | Collaborative | GMF branch + MLP branch combined |
| **Neural CF** | Collaborative | Concat embeddings → MLP (non-linear CF) |
| **ContentNet** | Content | User taste + genre/year (no movie ID embedding) |
| **HybridNet** | Hybrid | User + movie embeddings **+** content → MLP |

**How to use this notebook**

- **Default (`RUN_TRAINING = False`)** — read model explanations and load metrics from `artifacts/pipeline_report.json`. No GPU/CPU training required.
- **Interactive (`RUN_TRAINING = True`)** — train every model on a **300k-row sample** (~minutes per model) to reproduce the workflow locally.
- **Production run** — train on the full dataset with resume support:

  ```powershell
  python scripts/train_pipeline.py
  ```

**Data rule:** drop rows with NA in required columns and duplicate `(userId, movieId)` pairs before any training.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACTS_DIR = ROOT / "artifacts"
sys.path.insert(0, str(ROOT))

import importlib
import scripts.model_helpers as mh
import scripts.neural_models as nm
importlib.reload(mh)
importlib.reload(nm)

TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VAL_PATH = PROCESSED_DIR / "val.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"
PIPELINE_REPORT = ARTIFACTS_DIR / "pipeline_report.json"
BEST_MODEL_PATH = ARTIFACTS_DIR / "best_model.pt"

# False = show saved full-pipeline metrics. True = train on notebook sample.
RUN_TRAINING = False

TRAIN_SAMPLE = 300_000
VAL_SAMPLE = 50_000
BATCH_SIZE = 4096
EPOCHS = 5
EMBED_DIM = 64
LR = 1e-3
RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REQUIRED_COLS = ("userId", "movieId", "rating", "genres", "release_year")

torch.manual_seed(RANDOM_STATE)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


def load_pipeline_report() -> dict:
    if not PIPELINE_REPORT.exists():
        raise FileNotFoundError(
            f"Missing {PIPELINE_REPORT}. Run: python scripts/train_pipeline.py"
        )
    with open(PIPELINE_REPORT, encoding="utf-8") as f:
        return json.load(f)


def pipeline_row(report: dict, model_name: str) -> pd.Series:
    rows = [r for r in report["all_models"] if r["model"] == model_name]
    if not rows:
        raise KeyError(f"Model not found in pipeline report: {model_name}")
    return pd.Series(rows[0])


def show_pipeline_metrics(model_name: str, report: dict | None = None) -> pd.DataFrame:
    report = report or load_pipeline_report()
    row = pipeline_row(report, model_name)
    table = pd.DataFrame(
        {
            "split": ["val", "test"],
            "RMSE": [row["val_RMSE"], row["test_RMSE"]],
            "MAE": [row["val_MAE"], row["test_MAE"]],
        }
    )
    best = report["best"]
    print(
        f"Full pipeline ({best['train_rows']:,} train rows, "
        f"{best['elapsed_seconds'] / 3600:.1f} h on {best['device']}):"
    )
    display(table)
    return table


def train_or_show(
    model_name: str,
    model,
    *,
    model_type: str,
    train_df,
    val_eval,
    mappings,
    content_lookup=None,
    results: list,
    histories: dict,
    report: dict | None = None,
) -> None:
    if RUN_TRAINING:
        print(f"Training {model_name} on {len(train_df):,}-row sample...")
        row, history = nm.evaluate_model(
            model_name,
            model,
            train_df,
            val_eval,
            mappings,
            model_type=model_type,
            content_lookup=content_lookup,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            lr=LR,
            device=DEVICE,
        )
        results.append(row)
        histories[model_name] = history
        print(f"  sample val RMSE = {row['RMSE']:.4f}  MAE = {row['MAE']:.4f}")
    else:
        show_pipeline_metrics(model_name, report=report)


mode = "notebook sample training" if RUN_TRAINING else "saved full-pipeline metrics"
print(f"Device: {DEVICE}  |  Mode: {mode}")

## 0 — Data quality: drop NAs & duplicates

1. Scan full parquet for missing values
2. Sample for notebook training (when `RUN_TRAINING = True`)
3. **Drop** any row with NA in `userId`, `movieId`, `rating`, `genres`, `release_year`
4. **Drop** duplicate `(userId, movieId)` — keep the latest rating (`keep='last'`)
5. Assert the cleaned frame has **zero** NA and **zero** duplicates

In [ ]:
for name, path in [("train", TRAIN_PATH), ("val", VAL_PATH)]:
    audit = mh.audit_parquet_missing(path)
    print(f"\n{name}: {audit['rows']:,} rows")
    for col, n in sorted(audit["na"].items()):
        if n:
            print(f"  NA {col}: {n:,} ({100 * n / audit['rows']:.4f}%)")

In [ ]:
MODEL_COLS = list(REQUIRED_COLS) + ["timestamp", "director", "cast"]

train_raw = mh.sample_parquet(TRAIN_PATH, TRAIN_SAMPLE, MODEL_COLS, seed=RANDOM_STATE)
val_raw = mh.sample_parquet(VAL_PATH, VAL_SAMPLE, MODEL_COLS, seed=RANDOM_STATE)

train_df, train_stats = mh.clean_ratings_df(train_raw, required_cols=REQUIRED_COLS)
val_df, val_stats = mh.clean_ratings_df(val_raw, required_cols=REQUIRED_COLS)

print("Train cleaning:", train_stats)
print("Val cleaning:", val_stats)

mh.assert_clean_ratings(train_df, required_cols=REQUIRED_COLS)
mh.assert_clean_ratings(val_df, required_cols=REQUIRED_COLS)
print("\nOK — no NA or duplicates in cleaned train/val samples.")

In [ ]:
movies_unique = train_df[["movieId", "genres", "release_year"]].drop_duplicates("movieId")
movie_features_arr, vocabulary = mh.build_movie_content_features(movies_unique)
movie_features = pd.DataFrame(movie_features_arr, index=movies_unique["movieId"].values)

example = train_df.loc[train_df["genres"].str.contains("|")].iloc[0]
print(f"Multi-genre example: {example['genres']} → {mh.split_genre_string(example['genres'])}")
print(f"Content features: {movie_features.shape[0]:,} movies × {movie_features.shape[1]} dims")

mappings = nm.build_id_mappings(train_df)
content_lookup = nm.build_content_lookup(mappings, movie_features)
val_eval = val_df[
    val_df["userId"].isin(mappings.user_to_idx)
    & val_df["movieId"].isin(mappings.movie_to_idx)
].copy()
mh.assert_clean_ratings(val_eval, required_cols=REQUIRED_COLS)
print(f"Val rows for eval (seen in train sample): {len(val_eval):,} / {len(val_df):,}")

results: list[dict] = []
histories: dict[str, list] = {}
pipeline_report = load_pipeline_report() if not RUN_TRAINING else None
if pipeline_report:
    best = pipeline_report["best"]
    print(
        f"\nLoaded pipeline report — best model: {best['selected_model']} "
        f"(val RMSE {best['val_RMSE']:.4f})"
    )

## 1 — Baseline (global mean)

Predict every rating as the **global mean** of the training set. No user or movie personalization — a sanity-check lower bound.

If a model cannot beat this, it is not learning useful patterns.

**Full-pipeline results** (22,373,885 train rows, temporal val/test split):

| split | RMSE | MAE |
|-------|------|-----|
| val | 1.0167 | 0.7713 |
| test | 0.9982 | 0.7511 |

In [ ]:
if RUN_TRAINING:
    global_mean = train_df["rating"].mean()
    baseline_pred = np.full(len(val_eval), global_mean)
    rmse, mae = mh.rmse_mae(val_eval["rating"].to_numpy(), baseline_pred)
    results.append({"model": "Baseline (global mean)", "RMSE": rmse, "MAE": mae})
    print(f"Sample val RMSE = {rmse:.4f}  MAE = {mae:.4f}")
else:
    show_pipeline_metrics("Baseline (global mean)", report=pipeline_report)

## 2 — GMF (Generalized Matrix Factorization)

A collaborative filtering model that learns a dense embedding vector
for each user and each movie from historical ratings.

**For a user-movie pair:**
   1. Look up the user embedding and movie embedding.
   2. Compute their element-wise interaction.
   3. Use a final dense layer to predict the rating.

GMF is a neural-network version of matrix factorization:
 it learns latent factors (hidden preferences and movie attributes)
 directly from user and movie IDs.

 **Strengths:**
   - Simple and computationally efficient.
   - Learns meaningful user/movie embeddings.

 **Limitations:**
   - Uses only IDs (no genres, year, or other side features).
   - Captures limited nonlinear interactions compared to deeper models.

**Full-pipeline results:**

| split | RMSE | MAE |
|-------|------|-----|
| val | 1.5170 | 1.1160 |
| test | 1.5287 | 1.1292 |

In [ ]:
gmf = nm.GMF(mappings.n_users, mappings.n_movies, embed_dim=EMBED_DIM)
train_or_show(
    "GMF",
    gmf,
    model_type="cf",
    train_df=train_df,
    val_eval=val_eval,
    mappings=mappings,
    results=results,
    histories=histories,
    report=pipeline_report,
)

## 3 — NeuMF (Neural Matrix Factorization)

Combines two collaborative filtering approaches:

   **1. GMF branch:** 

      Learns user/movie embeddings and captures linear interactions
      through element-wise multiplication.

   **2. MLP branch:**

      Concatenates user/movie embeddings and passes them through
      hidden layers to learn complex nonlinear interactions.

 The outputs of both branches are combined to predict the rating.

 This allows NeuMF to capture both simple preference matching and
 deeper user-movie relationship patterns.

 **Pros:** More expressive and typically more accurate than GMF alone.
 
 **Cons:** More parameters and training time than simpler CF models.

**Full-pipeline results:**

| split | RMSE | MAE |
|-------|------|-----|
| val | 0.8230 | 0.6113 |
| test | 0.8329 | 0.6166 |

In [ ]:
neumf = nm.NeuMF(mappings.n_users, mappings.n_movies, gmf_dim=32, mlp_dim=32)
train_or_show(
    "NeuMF",
    neumf,
    model_type="cf",
    train_df=train_df,
    val_eval=val_eval,
    mappings=mappings,
    results=results,
    histories=histories,
    report=pipeline_report,
)

## 4 — Neural CF (MLP)

 A collaborative filtering model that learns embedding vectors for
 users and movies, concatenates them, and passes them through a
 multi-layer perceptron **(MLP)** to predict ratings.

 Unlike **GMF, Neural CF** does not use a dot product or explicit
 interaction term. Instead, the neural network learns complex
 nonlinear relationships directly from user and movie embeddings.

 **Pros:**
  Captures rich user-movie interaction patterns and often
 achieves strong recommendation performance.
 **Cons:**
  More computationally expensive and less interpretable than
 traditional matrix factorization methods.

Often competitive with NeuMF when interaction data is dense (as in MovieLens 32M).

**Full-pipeline results:**

| split | RMSE | MAE |
|-------|------|-----|
| val | 0.8249 | 0.6125 |
| test | 0.8326 | 0.6161 |

In [ ]:
neural_cf = nm.NeuralCF(mappings.n_users, mappings.n_movies, embed_dim=EMBED_DIM)
train_or_show(
    "Neural CF (MLP)",
    neural_cf,
    model_type="cf",
    train_df=train_df,
    val_eval=val_eval,
    mappings=mappings,
    results=results,
    histories=histories,
    report=pipeline_report,
)

## 5 — ContentNet (metadata only)

```
userId → Embedding ──┐
                     ├─ concat → MLP → rating
genre multi-hot+year ┘
```

 A content-based recommendation model that combines a user embedding
 with movie metadata features **(genres and release year)** to predict ratings.

 Unlike collaborative filtering models, ContentNet does not use movie
 ID embeddings or historical user-movie interactions. Instead, it
 learns preferences based on movie characteristics.

 **Pros:**
  Handles cold-start movies well because predictions can be made
 using metadata alone.
 
 ***Cons:***
  Cannot learn popularity effects or collaborative patterns from
 user behavior, so performance is typically lower on well-rated movies.

**Full-pipeline results:**

| split | RMSE | MAE |
|-------|------|-----|
| val | 0.9068 | 0.6779 |
| test | 0.9102 | 0.6797 |

In [ ]:
content_model = nm.ContentNet(
    mappings.n_users,
    content_dim=content_lookup.shape[1],
    embed_dim=EMBED_DIM,
)
train_or_show(
    "ContentNet",
    content_model,
    model_type="content",
    train_df=train_df,
    val_eval=val_eval,
    mappings=mappings,
    content_lookup=content_lookup,
    results=results,
    histories=histories,
    report=pipeline_report,
)

## 6 — HybridNet (collaborative + content)

Main **CineMatch** architecture. Feeds **user embedding**, **movie embedding**, and **content features** into one MLP.

Learns collaborative patterns (who rated what) **and** content affinity (genres, era) jointly. Selected as the **best model** on validation RMSE.

**Full-pipeline results:**

| split | RMSE | MAE |
|-------|------|-----|
| val | **0.8214** | **0.6094** |
| test | **0.8303** | **0.6143** |

In [ ]:
hybrid = nm.HybridNet(
    mappings.n_users,
    mappings.n_movies,
    content_dim=content_lookup.shape[1],
    embed_dim=EMBED_DIM,
)
train_or_show(
    "HybridNet",
    hybrid,
    model_type="hybrid",
    train_df=train_df,
    val_eval=val_eval,
    mappings=mappings,
    content_lookup=content_lookup,
    results=results,
    histories=histories,
    report=pipeline_report,
)

## 7 — Compare all models

Full-pipeline ranking (sorted by val RMSE). HybridNet wins by a small margin over NeuMF and Neural CF.

In [ ]:
if RUN_TRAINING:
    summary = pd.DataFrame(results).sort_values("RMSE")
    summary = summary.rename(columns={"RMSE": "val_RMSE", "MAE": "val_MAE"})
    title = "Notebook sample (val only)"
else:
    summary = pd.DataFrame(pipeline_report["all_models"]).sort_values("val_RMSE")
    title = "Full pipeline (val + test)"

display(summary)

fig, ax = plt.subplots(figsize=(10, 4))
summary.plot(
    x="model",
    y="val_RMSE",
    kind="bar",
    ax=ax,
    legend=False,
    color="steelblue",
)
ax.set_title(f"{title} — val RMSE (lower is better)")
ax.set_ylabel("val RMSE")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

if RUN_TRAINING and histories:
    fig, ax = plt.subplots(figsize=(10, 4))
    for name, hist in histories.items():
        ax.plot(pd.DataFrame(hist)["epoch"], pd.DataFrame(hist)["val_rmse"], marker="o", label=name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val RMSE")
    ax.set_title("Training curves (notebook sample)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 8 — Other neural models (future pipeline)

| Model | When to use |
|-------|-------------|
| **Two-tower** | Retrieval at scale — separate user/movie encoders, dot product in ANN/FAISS |
| **DeepFM / xDeepFM** | Many categorical side features (director, decade buckets) |
| **AutoRec** | Autoencoder on rating matrix — good when data is very sparse |
| **Transformer (SASRec)** | Sequential sessions — predict next rating from watch history order |
| **Graph NN (PinSage)** | Social / similarity graph between movies |
| **MiniLM + HybridNet** | Replace genre/year with neural text embeddings of overview (Phase 2) |

For explicit **star ratings** with metadata, **NeuMF + HybridNet** is the standard starting point before scaling up.

## 9 — Recommendation demo (HybridNet movie embeddings)

Cosine similarity over learned **movie embeddings** from the saved best checkpoint (`artifacts/best_model.pt`). Works in read-only mode — no retraining required.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

if RUN_TRAINING:
    demo_model = hybrid
    demo_mappings = mappings
else:
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Missing {BEST_MODEL_PATH}. Run: python scripts/train_pipeline.py"
        )
    ckpt = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
    demo_model = nm.build_model_from_checkpoint(ckpt)
    demo_mappings = nm.id_mappings_from_checkpoint(ckpt)
    print(f"Loaded checkpoint: {ckpt['model_name']} (val RMSE {ckpt['metrics']['val_RMSE']:.4f})")

titles = pd.read_parquet(TRAIN_PATH, columns=["movieId", "title"]).drop_duplicates("movieId")
title_by_id = titles.set_index("movieId")["title"].to_dict()

demo_model.eval()
with torch.no_grad():
    movie_emb = demo_model.movie_emb.weight.cpu().numpy()

seed_ids = train_df["movieId"].drop_duplicates().head(3).tolist()
seed_idx = [demo_mappings.movie_to_idx[i] for i in seed_ids if i in demo_mappings.movie_to_idx]
seed_ids = [demo_mappings.movie_ids[i] for i in seed_idx]
print("Seeds:", [title_by_id.get(i, i) for i in seed_ids])

profile = movie_emb[seed_idx].mean(axis=0, keepdims=True)
sims = cosine_similarity(profile, movie_emb).flatten()

rec = pd.DataFrame({"movie_idx": np.arange(len(sims)), "similarity": sims})
rec["movieId"] = demo_mappings.movie_ids[rec["movie_idx"]]
rec = rec[~rec["movieId"].isin(seed_ids)].sort_values("similarity", ascending=False).head(10)
rec["title"] = rec["movieId"].map(title_by_id)
rec[["title", "similarity"]]